# 📊 Tüm BIST (~606 Hisse) HMM+MOM3 — Düzeltilmiş Giriş Zamanlaması
## Veri: TradingView (tvdatafeed) | Evren: Tüm BIST | Timeframe: Haftalık

**Temel Düzeltme:** Zirvedeki hisseyi değil, **boğa trendinde geri çekilmiş** hisseyi sinyal ver.

| Eski Sorun | Yeni Çözüm |
|---|---|
| Tüm lagging göstergeler yükselen hisseyi işaretler | Overextension + RSI + 52H zirve cezaları eklendi |
| MOM yüksek = daha iyi skor | MOM ideal aralık: %10-60 (parabolic hareketler cezalı) |
| Zirvedeki hisse GÜÇLÜ AL verir | Zirveye yakın + RSI>70 → AŞIRI DEĞER uyarısı |
| Geç giriş riski | DİP FIRSATI sinyali: boğa rejiminde pullback = en iyi giriş |

**Sinyal Hiyerarşisi (En İyi → En Kötü):**  
`DİP FIRSATI` > `GÜÇLÜ AL` > `AL` > `HMM AL` > `MOM AL` > `BEKLE` > `AŞIRI DEĞER`


In [ ]:
import subprocess, sys

def pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg, imp in [
    ("git+https://github.com/rongardF/tvdatafeed.git", "tvDatafeed"),
    ("hmmlearn", "hmmlearn"),
    ("tqdm", "tqdm"),
]:
    try:
        __import__(imp)
    except ImportError:
        print(f"{imp} kuruluyor...")
        pip(pkg)

print("✅ Tüm kütüphaneler hazır.")


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import requests, time, os, pickle
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.linalg import inv
from hmmlearn.hmm import GaussianHMM
from tvDatafeed import TvDatafeed, Interval
from tqdm.notebook import tqdm

pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 170)
print("✅ Import'lar tamam.")


In [ ]:
INTERVAL   = Interval.in_weekly
N_BARS     = 200          # ~4 yıl haftalık veri
EXCHANGE   = "BIST"
BENCHMARK  = "XU100"

CACHE_DATA  = True
CACHE_FILE  = "bist_weekly_cache.pkl"

MIN_BARS    = 52
MAX_SYMBOLS = 700
TRAIN_MIN   = 52
TEST_SIZE   = 13
STEP        = 13

# TradingView hesabı (opsiyonel)
TV_USERNAME = ""
TV_PASSWORD = ""

# ── Giriş Zamanlaması Filtreleri (ANA DÜZELTME) ──────────────────────────
RSI_OVERBOUGHT   = 70    # RSI bu değerin üstündeyse AL verme → AŞIRI DEĞER
RSI_IDEAL_MAX    = 65    # İdeal alım bölgesi üst sınırı
OVEREXT_THRESH   = 0.20  # Fiyat SMA26'nın %20+ üzerindeyse penaltı başlar
TOP_PROXIMITY    = 0.05  # 52 haftalık zirveye %5'ten yakınsa penaltı
MOM13_IDEAL_MIN  = 0.05  # 13H momentum ideal alt sınırı (%5)
MOM13_IDEAL_MAX  = 0.70  # 13H momentum ideal üst sınırı (%70) — üstü parabolic

print("✅ Konfigürasyon yüklendi.")
print(f"   RSI overbought eşiği : {RSI_OVERBOUGHT}")
print(f"   Overextension eşiği  : %{OVEREXT_THRESH*100:.0f} üzeri SMA26")
print(f"   İdeal MOM13W aralığı : %{MOM13_IDEAL_MIN*100:.0f} – %{MOM13_IDEAL_MAX*100:.0f}")


In [ ]:
tv = TvDatafeed(TV_USERNAME, TV_PASSWORD) if TV_USERNAME else TvDatafeed()

try:
    _t = tv.get_hist("THYAO", EXCHANGE, interval=INTERVAL, n_bars=5)
    print(f"✅ TvDatafeed bağlantısı OK — THYAO {len(_t)} bar")
except Exception as e:
    print(f"❌ Bağlantı hatası: {e}")


In [ ]:
def get_all_bist():
    url = "https://scanner.tradingview.com/turkey/scan"
    payload = {
        "filter": [{"left":"type","operation":"equal","right":"stock"}],
        "options": {"lang":"tr"},
        "symbols": {"query":{"types":["stock"]},"tickers":[]},
        "columns": ["name","close","volume","market_cap_basic","average_volume_10d_calc"],
        "sort": {"sortBy":"market_cap_basic","sortOrder":"desc"},
        "range": [0, MAX_SYMBOLS],
    }
    try:
        r = requests.post(url, json=payload, timeout=30,
                          headers={"User-Agent":"Mozilla/5.0"})
        data = r.json().get("data", [])
        syms = []
        for row in data:
            s = row.get("s","")
            if s.startswith("BIST:"):
                syms.append(s.split(":")[1])
        print(f"✅ TradingView Scanner: {len(syms)} BIST sembolü")
        return syms
    except Exception as e:
        print(f"❌ Scanner hatası: {e} — yedek liste kullanılıyor")
        return None

ALL_SYMBOLS = get_all_bist()
if ALL_SYMBOLS is None:
    ALL_SYMBOLS = [
        "THYAO","GARAN","ASELS","BIMAS","EREGL","KCHOL","AKBNK","TUPRS",
        "FROTO","SISE","HALKB","VAKBN","MGROS","ASTOR","TKFEN","ISCTR",
        "TOASO","CCOLA","ENKAI","SAHOL","YKBNK","TCELL","PETKM","PGSUS",
        "KOZAL","OYAKC","ARCLK","KRDMD","DOHOL","SASA","TTKOM","AGHOL",
        "ULKER","AEFES","EKGYO","TAVHL","MAVI","BRSAN","GESAN","CWENE",
        "EUPWR","FENER","MIATK","PATEK","QUAGR","KTLEV","CVKMD","VESTL",
    ]

print(f"Toplam taranacak: {len(ALL_SYMBOLS)} hisse")


In [ ]:
def safe_get(sym, retries=3):
    for i in range(retries):
        try:
            df = tv.get_hist(sym, EXCHANGE, interval=INTERVAL, n_bars=N_BARS)
            if df is not None and len(df) >= MIN_BARS:
                df.columns = [c.lower() for c in df.columns]
                return df.sort_index()[~df.sort_index().index.duplicated()]
        except: time.sleep(1.5**i)
    return None

RAW = {}
if CACHE_DATA and os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "rb") as f: RAW = pickle.load(f)
    print(f"📂 Önbellekten yüklendi: {len(RAW)} hisse")
    to_dl = [s for s in ALL_SYMBOLS if s not in RAW]
else:
    to_dl = ALL_SYMBOLS

print("\nXU100 indiriliyor...")
xu100_raw = safe_get(BENCHMARK)
XU100 = xu100_raw["close"].dropna() if xu100_raw is not None else None
print(f"✅ XU100: {len(XU100) if XU100 is not None else 0} bar")

if to_dl:
    failed = []
    print(f"\n{len(to_dl)} hisse indiriliyor...")
    for sym in tqdm(to_dl, desc="Download"):
        df = safe_get(sym)
        if df is not None: RAW[sym] = df
        else: failed.append(sym)
        time.sleep(0.25)
    if CACHE_DATA:
        with open(CACHE_FILE,"wb") as f: pickle.dump(RAW, f)
        print(f"💾 Cache kaydedildi: {CACHE_FILE}")
    print(f"✅ Başarılı: {len(RAW)} | Başarısız: {len(failed)}")


In [ ]:
def prepare(df_raw, xu100=None):
    df = df_raw[["open","high","low","close","volume"]].copy()
    df["close"] = pd.to_numeric(df["close"], errors="coerce")
    df.dropna(subset=["close"], inplace=True)
    if len(df) < MIN_BARS: return None

    df["log_ret"] = np.log(df["close"] / df["close"].shift(1))
    # Split tespiti
    df.loc[df["log_ret"] < -0.45, "log_ret"] = np.nan

    # Göreceli güç
    if xu100 is not None:
        xu = xu100.reindex(df.index, method="ffill")
        df["rel_xu100"] = df["log_ret"] - np.log(xu/xu.shift(1))
    else:
        df["rel_xu100"] = df["log_ret"] - df["log_ret"].rolling(26).mean()

    # Momentum
    df["mom3w"]  = df["close"].pct_change(3)
    df["mom13w"] = df["close"].pct_change(13)

    # Hareketli ortalamalar
    df["sma13"] = df["close"].rolling(13).mean()
    df["sma26"] = df["close"].rolling(26).mean()

    # 52 haftalık yüksek/düşük (overextension tespiti için)
    df["high52w"] = df["close"].rolling(52).max()
    df["low52w"]  = df["close"].rolling(52).min()

    # Fiyatın SMA26'dan uzaklığı
    df["ext_ratio"] = df["close"] / df["sma26"].clip(lower=1e-9) - 1

    # RSI(14)
    d = df["close"].diff()
    df["rsi14"] = 100 - 100/(1 + d.clip(lower=0).rolling(14).mean() /
                              (-d.clip(upper=0)).rolling(14).mean().clip(lower=1e-9))

    # ATR(14)
    tr = pd.concat([
        df["high"]-df["low"],
        (df["high"]-df["close"].shift(1)).abs(),
        (df["low"]-df["close"].shift(1)).abs(),
    ], axis=1).max(axis=1)
    df["atr14"] = tr.rolling(14).mean()

    df["vol_ratio"] = df["volume"] / df["volume"].rolling(20).mean().clip(lower=1)
    return df

WEEKLY = {}
for t, raw in RAW.items():
    try:
        df = prepare(raw, XU100)
        if df is not None: WEEKLY[t] = df
    except: pass

print(f"✅ {len(WEEKLY)} hisse için indikatörler hazır.")


In [ ]:
class HMMStrategy:
    def __init__(self, n_iter=300, min_samples=30):
        self.n_iter, self.min_samples = n_iter, min_samples
        self.model, self.bull_state = None, 0

    def _X(self, df):
        return df[["log_ret","rel_xu100"]].replace([np.inf,-np.inf],np.nan).dropna()

    def fit(self, df):
        X = self._X(df)
        if len(X) < self.min_samples: return False
        try:
            m = GaussianHMM(n_components=2, covariance_type="full",
                            n_iter=self.n_iter, random_state=42)
            m.fit(X.values)
            self.model = m
            self.bull_state = int(np.argmax(m.means_[:,0]))
            return True
        except: return False

    def predict_states(self, df):
        if not self.model: return pd.Series(0, index=df.index)
        X = self._X(df)
        if X.empty: return pd.Series(0, index=df.index)
        try:
            s = self.model.predict(X.values)
            return pd.Series((s==self.bull_state).astype(float),
                             index=X.index).reindex(df.index, fill_value=0)
        except: return pd.Series(0, index=df.index)

    def bull_prob(self, df):
        if not self.model: return 0.5
        X = self._X(df)
        if X.empty: return 0.5
        try:
            return float(self.model.predict_proba(X.values)[-1, self.bull_state])
        except: return 0.5

    def weeks_in_current_regime(self, df):
        """Son rejime kaç haftadır devam ediliyor?"""
        states = self.predict_states(df)
        cur = int(states.iloc[-1])
        count = 0
        for v in reversed(states.values):
            if int(v) == cur: count += 1
            else: break
        return count


def wfo_hmm(df):
    n = len(df)
    oos_r, oos_d = [], []
    for fs in range(TRAIN_MIN, n - TEST_SIZE + 1, STEP):
        h = HMMStrategy()
        if not h.fit(df.iloc[:fs]): continue
        sig = h.predict_states(df.iloc[:fs+TEST_SIZE])
        sig_t = sig.iloc[fs:fs+TEST_SIZE].shift(1).fillna(0)
        ret_t = df["log_ret"].iloc[fs:fs+TEST_SIZE].fillna(0)
        oos_r.extend((sig_t*ret_t).tolist())
        oos_d.extend(ret_t.index.tolist())
    if not oos_r: return {"sharpe":-99, "ret":-99}
    s = pd.Series(oos_r, index=oos_d).sort_index()
    return {"sharpe": round(float(s.mean()/(s.std()+1e-9))*np.sqrt(52),3),
            "ret":    round(float(np.exp(s.sum())-1)*100, 1)}


def kalman_proj(close_s, n_fw=13):
    y = np.log(close_s.dropna().values)
    if len(y) < 20:
        p = float(close_s.iloc[-1]); return 0, p, p, p
    F=np.array([[1,1],[0,1]]); H=np.array([[1,0]])
    Q=np.eye(2)*1e-4; R=np.array([[1e-2]])
    x=np.array([[y[0]],[0.]]); P=np.eye(2)
    for obs in y:
        xp=F@x; Pp=F@P@F.T+Q; inn=obs-(H@xp)[0,0]
        S=H@Pp@H.T+R; K=Pp@H.T@inv(S); x=xp+K*inn; P=(np.eye(2)-K@H)@Pp
    vel=float(x[1,0]); lev=float(x[0,0])
    fl=lev+vel*n_fw; sig=float(np.std(np.diff(y))*np.sqrt(n_fw))
    return (round(vel*52*100,2), round(float(np.exp(fl)),2),
            round(float(np.exp(fl-2*sig)),2), round(float(np.exp(fl+2*sig)),2))

print("✅ HMM | WFO | Kalman modelleri hazır.")


In [ ]:
def scan_ticker(ticker):
    df = WEEKLY.get(ticker)
    if df is None: return None
    df = df.dropna(subset=["log_ret","rel_xu100"])
    if len(df) < MIN_BARS: return None

    # ── Temel değerler ──────────────────────────────────────────────────────
    cur   = float(df["close"].iloc[-1])
    atr   = float(df["atr14"].iloc[-1])    if pd.notna(df["atr14"].iloc[-1])    else cur*0.03
    rsi   = float(df["rsi14"].iloc[-1])    if pd.notna(df["rsi14"].iloc[-1])    else 50
    vol   = float(df["vol_ratio"].iloc[-1]) if pd.notna(df["vol_ratio"].iloc[-1]) else 1.
    mom13 = float(df["mom13w"].iloc[-1])   if pd.notna(df["mom13w"].iloc[-1])   else 0
    mom3  = float(df["mom3w"].iloc[-1])    if pd.notna(df["mom3w"].iloc[-1])    else 0
    sma13 = float(df["sma13"].iloc[-1])    if pd.notna(df["sma13"].iloc[-1])    else cur
    sma26 = float(df["sma26"].iloc[-1])    if pd.notna(df["sma26"].iloc[-1])    else cur
    ext   = float(df["ext_ratio"].iloc[-1]) if pd.notna(df["ext_ratio"].iloc[-1]) else 0
    h52   = float(df["high52w"].iloc[-1])  if pd.notna(df["high52w"].iloc[-1])  else cur

    wfo = wfo_hmm(df)

    # ── HMM ─────────────────────────────────────────────────────────────────
    hmm = HMMStrategy()
    hmm.fit(df)
    prob         = hmm.bull_prob(df)
    hmm_sig      = prob > 0.55
    weeks_in_bull = hmm.weeks_in_current_regime(df) if hmm_sig else 0

    # ── Kalman ──────────────────────────────────────────────────────────────
    vel_pct, tgt, tgt_lo, tgt_hi = kalman_proj(df["close"])
    upside = (tgt / cur - 1) * 100

    # ── Giriş kalitesi metrikleri ────────────────────────────────────────────
    top_dist_pct = (h52 - cur) / h52 * 100        # Zirveye % uzaklık
    ext_pct      = ext * 100                        # SMA26'dan % uzama
    is_pullback  = mom3 < -0.02 and mom13 > MOM13_IDEAL_MIN  # Boğada dip
    is_fresh     = 0 < weeks_in_bull <= 4            # Taze boğa sinyali
    is_parabolic = mom13 > MOM13_IDEAL_MAX           # Çok hızlı yükseliş

    # ── Tükenme (Exhaustion) Tespiti ────────────────────────────────────────
    # Kullanıcı notu: RSI>70 YALNIZ BAŞINA "aşırı değer" DEĞİLDİR.
    # Güçlü bir boğa trendinde RSI haftalarca 70+ kalabilir (örn: ASTOR 2023-24).
    # Gerçek tükenme/dağıtım = RSI yüksek + momentum yavaşlıyor + hacim azalıyor
    #
    # Momentum yavaşlaması: 3H ivme, 13H ivmenin %30'undan düşük
    # → fiyat hâlâ yükseliyor ama hızı giderek azalıyor (gizli bearish divergence)
    mom_slowing   = (mom3 < mom13 * 0.30) and (mom13 > 0)

    # Hacim azalması: son hafta hacmi 20H ortalamasının %75'inden az
    # → fiyat zirve yaparken alıcı katılımı düşüyor (dağıtım paterni)
    vol_declining = vol < 0.75

    # AŞIRI DEĞER için en az 1 kombinasyon gerekli:
    #   A) RSI>70  VE  (momentum yavaşlıyor  VEYA  hacim azalıyor)
    #   B) Parabolic  VE  (momentum yavaşlıyor  VEYA  hacim azalıyor)
    #   C) Çok aşırı uzamış (>%35) VE RSI>65
    cond_a = (rsi > RSI_OVERBOUGHT) and (mom_slowing or vol_declining)
    cond_b = is_parabolic            and (mom_slowing or vol_declining)
    cond_c = (ext_pct > 35)          and (rsi > 65)
    is_overvalued = cond_a or cond_b or cond_c

    # ── Sinyal Mantığı ──────────────────────────────────────────────────────
    if   is_overvalued and hmm_sig:
        sinyal = "AŞIRI DEĞER"   # Boğa ama tükenme sinyalleri var → bekle
    elif hmm_sig and is_pullback and rsi < RSI_OVERBOUGHT:
        sinyal = "DİP FIRSATI"   # En iyi giriş: boğa trendinde pullback
    elif hmm_sig and MOM13_IDEAL_MIN < mom13 <= MOM13_IDEAL_MAX \
         and rsi < RSI_OVERBOUGHT and ext_pct < 20 and sma13 > sma26:
        sinyal = "GÜÇLÜ AL"      # Tüm koşullar ideal
    elif hmm_sig and mom13 > 0 and rsi < RSI_OVERBOUGHT and ext_pct < OVEREXT_THRESH*100:
        sinyal = "AL"
    elif hmm_sig and rsi < RSI_OVERBOUGHT:
        sinyal = "HMM AL"
    elif mom13 > MOM13_IDEAL_MIN and sma13 > sma26 and rsi < RSI_OVERBOUGHT:
        sinyal = "MOM AL"
    else:
        sinyal = "BEKLE"

    # ── Kompozit Skor ────────────────────────────────────────────────────────
    # HMM güven: 0-40 puan
    s_hmm  = prob * 40

    # Momentum: ideal aralıkta (5-70%) en yüksek puan
    if MOM13_IDEAL_MIN <= mom13 <= MOM13_IDEAL_MAX:
        s_mom = 20
    elif mom13 < MOM13_IDEAL_MIN:
        s_mom = max(0, mom13 / MOM13_IDEAL_MIN * 20)
    else:
        excess = (mom13 - MOM13_IDEAL_MAX) / MOM13_IDEAL_MAX
        s_mom  = max(0, 20 - excess * 40)   # Parabolic ceza

    # Kalman hız: yıllık %0-40 arası ödül (0-15 puan)
    s_kal  = min(max(vel_pct / 40 * 15, 0), 15)

    # Giriş kalitesi: overextension + RSI + zirve yakınlığı cezaları (0-15 puan)
    s_entry  = 15
    s_entry -= min(max(ext_pct - 10, 0) * 0.5, 10)        # Her %1 uzama → -0.5
    s_entry -= min(max(rsi - 55, 0) * 0.3, 8)             # RSI>55 → ceza
    s_entry -= max(0, (TOP_PROXIMITY*100 - top_dist_pct))  # Zirveye yakın ceza
    s_entry -= 5 if mom_slowing  else 0                    # Momentum yavaşlıyor
    s_entry -= 3 if vol_declining else 0                   # Hacim azalıyor
    s_entry  = max(0, s_entry)

    # MA trend (0-5 puan)
    s_ma  = 5 if sma13 > sma26 else 0

    # Hacim (0-5 puan) — yükselen hacim pozitif, düşen negatif
    s_vol = min(vol * 2.5, 5)

    # Bonuslar
    bonus  = 5 if is_pullback else 0   # Dip fırsatı
    bonus += 3 if is_fresh    else 0   # Taze boğa sinyali

    composite = max(0, min(100, s_hmm + s_mom + s_kal + s_entry + s_ma + s_vol + bonus))

    return dict(
        ticker=ticker, fiyat=cur, sinyal=sinyal,
        hmm_prob=round(prob*100, 1),
        wfo_sharpe=wfo["sharpe"], wfo_ret=wfo["ret"],
        mom13w=round(mom13*100, 1), mom3w=round(mom3*100, 1),
        rsi=round(rsi, 1),
        ext_pct=round(ext_pct, 1),
        top_dist=round(top_dist_pct, 1),
        mom_slowing=mom_slowing,
        vol_declining=vol_declining,
        vol_ratio=round(vol, 2),
        kal_vel=vel_pct, kal_hedef=tgt,
        kal_upside=round(upside, 1),
        weeks_bull=weeks_in_bull,
        pullback=is_pullback, fresh=is_fresh,
        stop=round(cur - 2*atr, 2),
        composite=round(composite, 1),
    )

print("✅ Tarama fonksiyonu (tükenme tespiti güncellenmiş) hazır.")
print()
print("AŞIRI DEĞER için en az 1 kriter gerekli:")
print("  A) RSI>70 VE (momentum yavaşlıyor VEYA hacim azalıyor)")
print("  B) Parabolic(MOM13>%70) VE (momentum yavaşlıyor VEYA hacim azalıyor)")
print("  C) Ext>%35 VE RSI>65")
print()
print("RSI>70 YALNIZ BAŞINA 'aşırı değer' değildir — güçlü trendde normal!")


In [ ]:
print(f"🔍 Tarama başlıyor: {len(WEEKLY)} hisse\n")

results, errors = [], []
for ticker in tqdm(list(WEEKLY.keys()), desc="HMM+WFO Tarama"):
    try:
        r = scan_ticker(ticker)
        if r: results.append(r)
    except Exception as e:
        errors.append((ticker, str(e)))

DF = (pd.DataFrame(results)
        .sort_values("composite", ascending=False)
        .reset_index(drop=True))
DF.index += 1

# Sinyal özeti
print("\n" + "="*60)
SINYAL_ORDER = ["DİP FIRSATI","GÜÇLÜ AL","AL","HMM AL","MOM AL","AŞIRI DEĞER","BEKLE"]
EMOJI = {"DİP FIRSATI":"⭐","GÜÇLÜ AL":"🟢","AL":"🟡","HMM AL":"🔵",
         "MOM AL":"🟣","AŞIRI DEĞER":"⚠️","BEKLE":"🔴"}
for s in SINYAL_ORDER:
    n = len(DF[DF["sinyal"]==s])
    if n: print(f"  {EMOJI.get(s,'')} {s:<15}: {n:>4} hisse")
print("="*60)


In [ ]:
COLS = ["ticker","fiyat","sinyal","hmm_prob","mom13w","rsi",
        "ext_pct","top_dist","kal_hedef","kal_upside","kal_vel",
        "wfo_sharpe","weeks_bull","stop","composite"]

date_str = pd.Timestamp.today().strftime("%Y-%m-%d")

print(f"{'='*115}")
print(f"📊 TÜM BIST HMM+MOM3 GİRİŞ ZAMANLAMALI TARAMA — {date_str}")
print(f"   ext_pct=SMA26'dan uzaklık% | top_dist=52H zirvesine uzaklık% | weeks_bull=boğa rejiminde hafta")
print(f"{'='*115}")

# ⭐ DİP FIRSATI — en iyi giriş noktaları
dip = DF[DF["sinyal"]=="DİP FIRSATI"]
print(f"\n⭐ DİP FIRSATI ({len(dip)} hisse) — Boğa trendinde pullback: EN İYİ GİRİŞ NOKTASI")
print("-"*115)
if len(dip): print(dip[COLS].to_string())

# 🟢 GÜÇLÜ AL
guclu = DF[DF["sinyal"]=="GÜÇLÜ AL"]
print(f"\n🟢 GÜÇLÜ AL ({len(guclu)} hisse) — İdeal koşullar, aşırı uzamamış")
print("-"*115)
if len(guclu): print(guclu[COLS].to_string())

# 🟡 AL
al = DF[DF["sinyal"]=="AL"]
print(f"\n🟡 AL ({len(al)} hisse) — HMM boğa + momentum, ext_pct<%20")
print("-"*115)
if len(al): print(al.head(20)[COLS].to_string())

# ⚠️ AŞIRI DEĞER — bilgi amaçlı
asiri = DF[DF["sinyal"]=="AŞIRI DEĞER"]
print(f"\n⚠️  AŞIRI DEĞER ({len(asiri)} hisse) — Boğa trendinde ama RSI>70 veya aşırı yükselmiş, GİRMEYİN")
print("-"*115)
if len(asiri): print(asiri.head(20)[["ticker","fiyat","hmm_prob","rsi","ext_pct","top_dist","mom13w","composite"]].to_string())

# Tam CSV
DF.to_csv(f"bist_tarama_{date_str}.csv", index=True)
print(f"\n💾 Kaydedildi: bist_tarama_{date_str}.csv")


In [ ]:
# En iyi giriş noktaları: DİP FIRSATI önce, sonra GÜÇLÜ AL
priority_sigs = ["DİP FIRSATI","GÜÇLÜ AL","AL","HMM AL"]
plot_df = pd.concat([DF[DF["sinyal"]==s] for s in priority_sigs]).head(20)
if len(plot_df) == 0: plot_df = DF.head(20)

n = len(plot_df); ncols = 5
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(24, nrows*4))
axes = axes.flatten()
fig.suptitle(
    f"BIST — Giriş Zamanlamalı HMM+MOM3 En İyi {n} Sinyal | {date_str}\n"
    "⭐=Dip Fırsatı 🟢=Güçlü AL | Sarı bölge=aşırı uzama eşiği | Mavi=Kalman hedef",
    fontsize=12, fontweight="bold"
)
hmm_cache = {}

for idx, (_, row) in enumerate(plot_df.iterrows()):
    ax = axes[idx]; tkr = row["ticker"]
    if tkr not in WEEKLY: ax.set_visible(False); continue
    df = WEEKLY[tkr].tail(104)
    if tkr not in hmm_cache:
        h = HMMStrategy(); h.fit(WEEKLY[tkr]); hmm_cache[tkr] = h
    states = hmm_cache[tkr].predict_states(WEEKLY[tkr]).reindex(df.index).fillna(0)

    ax.plot(df.index, df["close"], color="black", lw=1.3, zorder=5)
    for j in range(len(df)-1):
        c = "#d0f5d0" if states.iloc[j]==1 else "#f5d0d0"
        ax.axvspan(df.index[j], df.index[j+1], alpha=0.3, color=c, zorder=1)
    ax.plot(df.index, df["sma13"], color="darkorange", lw=0.9, alpha=0.85)
    ax.plot(df.index, df["sma26"], color="purple",     lw=0.9, alpha=0.85)

    # Aşırı uzama eşiği (SMA26 × 1.20) — sarı çizgi
    if "sma26" in df.columns:
        ax.plot(df.index, df["sma26"]*1.20, color="gold", lw=0.7,
                ls="--", alpha=0.7, label="+20%SMA")

    ax.axhline(row["kal_hedef"], color="royalblue", ls="--", lw=1.0)
    ax.axhline(row["stop"],      color="crimson",   ls=":",  lw=0.8)

    SIG_COLOR = {"DİP FIRSATI":"gold","GÜÇLÜ AL":"darkgreen",
                 "AL":"darkorange","HMM AL":"steelblue"}.get(row["sinyal"],"gray")
    em = {"DİP FIRSATI":"⭐","GÜÇLÜ AL":"🟢","AL":"🟡","HMM AL":"🔵"}.get(row["sinyal"],"")
    ax.set_title(
        f"{tkr} {em}{row['sinyal']}\n"
        f"Skor:{row['composite']:.0f} RSI:{row['rsi']:.0f} "
        f"Ext:{row['ext_pct']:+.0f}% Hedef:{row['kal_hedef']:.0f}({row['kal_upside']:+.0f}%)",
        fontsize=7.5, color=SIG_COLOR, fontweight="bold"
    )
    ax.tick_params(labelsize=6)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")

for i in range(idx+1, len(axes)): axes[i].set_visible(False)
plt.tight_layout()
plt.savefig(f"bist_giris_{date_str}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Grafik kaydedildi: bist_giris_{date_str}.png")


In [ ]:
print("\n" + "═"*80)
print("💼 QUARTER-KELLY POZİSYON ÖNERİLERİ")
print("   (Sadece DİP FIRSATI + GÜÇLÜ AL + AL sinyalleri)")
print("═"*80)

buy_df = DF[DF["sinyal"].isin(["DİP FIRSATI","GÜÇLÜ AL","AL"])].copy()

if len(buy_df) == 0:
    print("AL sinyali yok. HMM AL listesini inceleyin.")
    buy_df = DF[DF["sinyal"]=="HMM AL"].head(5)

pos_list = []
for _, r in buy_df.head(15).iterrows():
    p    = r["hmm_prob"]/100
    b    = max(r["kal_upside"]/100, 0.05)
    risk = max((r["fiyat"]-r["stop"])/r["fiyat"], 0.01)
    qk   = min(max((p*b-(1-p)*risk)/b, 0)*0.25, 0.15)
    pos_list.append({
        "Hisse":    r["ticker"],
        "Sinyal":   r["sinyal"],
        "Fiyat":    r["fiyat"],
        "Hedef":    r["kal_hedef"],
        "Stop":     r["stop"],
        "RSI":      r["rsi"],
        "Ext%":     r["ext_pct"],
        "ZirveUzk%":r["top_dist"],
        "R:R":      round(b/risk,2) if risk>0 else 0,
        "Pos%":     round(qk*100,1),
        "Skor":     r["composite"],
    })

if pos_list:
    pos_df = pd.DataFrame(pos_list).sort_values("Pos%", ascending=False)
    pos_df.reset_index(drop=True, inplace=True); pos_df.index+=1
    total = pos_df["Pos%"].sum()
    print(pos_df.to_string())
    print("-"*80)
    print(f"Toplam yatırım : %{total:.1f}  |  Nakit: %{100-total:.1f}")
    print("\n⚠️  Bu çıktı yatırım tavsiyesi değildir.")
